Calculate VaR/ES on 2 levels from simulated values - Copula, using test9_1_portfolio.csv, test9_1_returns.csv, and testout_9.1.csv

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import t, norm, multivariate_normal, spearmanr

In [2]:
data1 = pd.read_csv('../testfiles/data/test9_1_portfolio.csv')
data1

,Stock,Holding,Starting Price,Distribution
0,A,100,20,Normal
1,B,100,30,T


In [3]:
data2 = pd.read_csv('../testfiles/data/test9_1_returns.csv')
data2

,A,B
0,-0.008018,0.029859
1,-0.043707,-0.039298
2,-0.008501,-0.025579
3,-0.004756,-0.009689
4,0.003788,-0.006877
...,...,...
195,0.040275,0.009365
196,-0.028035,-0.004859
197,0.007395,0.015389
198,-0.034845,-0.020320


In [4]:
# A: normal dist
A_return = data2['A'].values
A_mu = np.mean(A_return)
A_sigma = np.std(A_return, ddof=1)

In [5]:
# B: T dist
B_return = data2['B'].values
B_nu, B_mu, B_sigma = t.fit(B_return)

In [6]:
# transform to uniforms
U_A = norm.cdf(A_return, A_mu, A_sigma)
U_B = t.cdf(B_return, df=B_nu, loc=B_mu, scale=B_sigma)
U = np.column_stack([U_A, U_B])

In [34]:
# Gaussian Copula corr
Z = norm.ppf(np.column_stack([U_A, U_B]))

# Pearson
# copula_corr = np.corrcoef(Z.T)

# Spearman
rho_s, _ = spearmanr(A_return, B_return)
copula_rho = 2 * np.sin(np.pi * rho_s / 6)
copula_corr = np.array([[1, copula_rho],
                        [copula_rho, 1]])

copula_corr

array([[1.        , 0.07349605],
       [0.07349605, 1.        ]])

In [35]:
# simulate joint uniforms
np.random.seed(42)
n_sims = 100_000
sim = multivariate_normal.rvs(np.zeros(2), copula_corr, n_sims)

U_sim = norm.cdf(sim)

In [36]:
# transform back to normal / T dist
A_sim = norm.ppf(U_sim[:, 0], A_mu, A_sigma)
B_sim = t.ppf(U_sim[:, 1], df=B_nu, loc=B_mu, scale=B_sigma)

R_sim = np.column_stack([A_sim, B_sim])

In [37]:
# convert return to P&L
A_pos_value = data1.loc[0, "Holding"] * data1.loc[0, "Starting Price"]
B_pos_value = data1.loc[1, "Holding"] * data1.loc[1, "Starting Price"]
V0 = A_pos_value + B_pos_value

# weighted portfolio return
A_weight = A_pos_value / V0
B_weight = B_pos_value / V0

weights = np.array([A_weight, B_weight])

A_loss = -A_sim * A_pos_value
B_loss = -B_sim * B_pos_value

R_port = A_weight * A_sim + B_weight * B_sim

In [38]:
# total VaR & ES
alpha = 0.05

def output_total(R_port, V0):
    q_alpha = np.quantile(R_port, alpha)
    
    var_pct = -q_alpha
    es_pct = -R_port[R_port <= q_alpha].mean()
    
    var = V0 * var_pct
    es = V0 * es_pct
    
    return var, es, var_pct, es_pct

total_var, total_es, total_var_pct, total_es_pct = output_total(R_port, V0)

In [39]:
# A VaR & ES
z_score = norm.ppf(alpha)
A_var_pct = abs(A_mu + z_score * A_sigma)
A_var = A_var_pct * A_pos_value

phi_z = norm.pdf(z_score)
A_es_pct = -A_mu + A_sigma * phi_z / alpha
A_es = A_es_pct * A_pos_value

In [40]:
# B VaR & ES
B_var_pct = abs(np.quantile(B_sim, alpha))
B_var = B_var_pct * B_pos_value

t_stat = t.ppf(alpha, df=B_nu)
phi_t = t.pdf(t_stat, df=B_nu)
B_es_pct = -B_mu + B_sigma * ((B_nu + t_stat**2) / (B_nu - 1)) * (phi_t / alpha)
B_es = B_es_pct * B_pos_value

In [41]:
print(f"Stock,VaR95,ES95,VaR95_Pct,ES95_Pct")
print(f"A, {A_var},{A_es}, {A_var_pct}, {A_es_pct}")
print(f"B, {B_var},{B_es}, {B_var_pct}, {B_es_pct}")
print(f"Total, {total_var},{total_es}, {total_var_pct}, {total_es_pct}")

Stock,VaR95,ES95,VaR95_Pct,ES95_Pct
A, 94.1999870886495,117.90588209324741, 0.047099993544324745, 0.05895294104662371
B, 109.28342770298596,152.2689495514617, 0.03642780923432865, 0.0507563165171539
Total, 152.21695182454565,198.40540849239014, 0.030443390364909128, 0.039681081698478025
